# Amazon ML Challenge 2026: Stage 2 — Pairwise Feature Engineering & LightGBM Scorer

**Input:** `candidates_train.parquet` from NB 02, normalized parquets from NB 00, ground truth TSV

**Pipeline position:** This is the **Matching Model** (PS Step 2). It scores every candidate pair and
produces calibrated probabilities to feed into the Decision Layer.

**Methodology reference:** Approach C5 (rerank with LightGBM on similarities + hand features),
A3 (feature engineering), A4 (training set construction), A5 (LightGBM), A6 (calibration).

**Outputs:**
- `score_C_train.parquet` — OOF predictions keyed by `(s1_id, candidate_id)` for threshold tuning
- `score_C_test.parquet`  — Ensemble-averaged test predictions for final submission
- `lgbm_model_fold{k}.txt` — Saved model checkpoints for reproducibility

**Requirements:** CPU notebook (no GPU needed). `lightgbm`, `rapidfuzz` pre-installed on Kaggle.

In [ ]:
!pip install -q rapidfuzz lightgbm

import os, sys, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from rapidfuzz import fuzz, distance
from rapidfuzz.distance import JaroWinkler

print("Libraries imported.")

# ============================================================
# HARDWARE CONFIGURATION
# ============================================================

USE_GPU = False
device_param = 'cpu'

try:
    import torch
    if torch.cuda.is_available():
        USE_GPU = True
        device_param = 'gpu'
        gpu_name = torch.cuda.get_device_name(0)
        n_gpus = torch.cuda.device_count()
except Exception:
    pass

if USE_GPU:
    print(f"🚀 GPU Acceleration: ENABLED ({n_gpus} GPU(s) available: {gpu_name})")
else:
    print("ℹ️ GPU Acceleration: OFF (Running on multithreaded CPU)")

# ============================================================
# PATH RESOLUTION (robust: handles Kaggle directory flattening)
# ============================================================

IS_KAGGLE = os.path.exists('/kaggle')

def find_dir_by_file(pattern):
    for p in Path('/kaggle/input').rglob(pattern):
        return p.parent
    return None

def find_norm_dir():
    # Method 1: rglob search for train_s1_norm.parquet
    found = find_dir_by_file('train_s1_norm.parquet')
    if found:
        return found, 'parquet'
    
    # Method 2: check known Kaggle notebook output paths (including ml1nbv2 & ml1nbv1)
    known_paths = [
        Path('/kaggle/input/ml1nbv2/normalized'),
        Path('/kaggle/input/ml1nbv2'),
        Path('/kaggle/input/notebooks/ojassangwai/ml1nbv2/normalized'),
        Path('/kaggle/input/notebooks/ojassangwai/ml1nbv2'),
        Path('/kaggle/input/ml1nbv1/normalized'),
        Path('/kaggle/input/ml1nbv1'),
        Path('/kaggle/input/notebooks/ojassangwai/ml1nbv1/normalized'),
        Path('/kaggle/input/notebooks/ojassangwai/ml1nbv1'),
    ]
    for p in known_paths:
        if (p / 'train_s1_norm.parquet').exists():
            return p, 'parquet'
    
    # Method 3: fallback to raw TSV files from the competition dataset
    data_dir = find_dir_by_file('train_ground_truth.tsv')
    if data_dir and (data_dir / 'train' / 'train_source1.tsv').exists():
        print("  ⚠️ Normalized parquets not found. Will normalize from raw TSVs (slower but works).")
        return data_dir, 'raw_tsv'
    
    # Diagnostic
    print("\n❌ ERROR: Could not find normalized parquets (train_s1_norm.parquet).")
    print("  Listing /kaggle/input data files for debugging:")
    for p in sorted(Path('/kaggle/input').rglob('*')):
        if p.is_file() and p.suffix in ('.parquet', '.tsv', '.csv', '.npy'):
            print(f"    {p}")
    return None, None

def find_candidates_file():
    # If multiple candidates_train.parquet exist (e.g. from ml2nbv1 and ml2nbv2),
    # prioritize ml2nbv2 over ml2nbv1 and ensure non-empty.
    cands = [p for p in Path('/kaggle/input').rglob('candidates_train.parquet') if p.is_file()]
    if not cands:
        return None
    
    def sort_key(p):
        path_str = str(p).lower()
        # Prefer v2 over v1
        v_score = 2 if 'v2' in path_str else (1 if 'v1' in path_str else 0)
        try:
            sz = p.stat().st_size
            mt = p.stat().st_mtime
        except Exception:
            sz, mt = 0, 0
        return (v_score, sz, mt)
    
    cands_sorted = sorted(cands, key=sort_key, reverse=True)
    best = cands_sorted[0]
    print(f"  Found {len(cands)} candidates_train.parquet candidate(s):")
    for cp in cands_sorted:
        tag = "[SELECTED]" if cp == best else "[IGNORED] "
        sz_mb = cp.stat().st_size / (1024 * 1024)
        print(f"    {tag} {cp} ({sz_mb:.1f} MB)")
    return best

if IS_KAGGLE:
    WORK_DIR = Path('/kaggle/working')
    
    # 1. candidates_train.parquet from Stage 1 (ml2nbv2 / ml2nbv1)
    CAND_FILE = find_candidates_file()
    if CAND_FILE is not None:
        CAND_DIR = CAND_FILE.parent
    else:
        CAND_DIR = WORK_DIR
        CAND_FILE = CAND_DIR / 'candidates_train.parquet'
    
    # 2. Normalized parquets from Stage 0 (ml1nbv2 / ml1nbv1)
    NORM_DIR, NORM_FORMAT = find_norm_dir()
    assert NORM_DIR is not None, (
        "Could not find normalized parquets!\n"
        "FIX: In the Kaggle notebook sidebar, click '+ Add Input' and attach\n"
        "     the output of your Stage 0 normalization notebook (ml1nbv2 or ml1nbv1).\n"
        "     It should contain: train_s1_norm.parquet, train_s2_norm.parquet, train_s3_norm.parquet"
    )
    
    # 3. Ground truth — exhaustive search
    GT_FILE = None
    # Method 1: rglob under all of /kaggle/input
    for p in Path('/kaggle/input').rglob('train_ground_truth.tsv'):
        GT_FILE = p
        break
    # Method 2: check known nested structures
    if GT_FILE is None:
        gt_candidates = [
            Path('/kaggle/input/mlchallengedata/student_resource/dataset/train/train_ground_truth.tsv'),
            Path('/kaggle/input/mlchallengedata/dataset/train/train_ground_truth.tsv'),
            Path('/kaggle/input/mlchallengedata/train/train_ground_truth.tsv'),
            Path('/kaggle/input/mlchallengedata/train_ground_truth.tsv'),
            Path('/kaggle/input/datasets/ojassangwai/mlchallengedata/student_resource/dataset/train/train_ground_truth.tsv'),
            Path('/kaggle/input/datasets/ojassangwai/mlchallengedata/dataset/train/train_ground_truth.tsv'),
            Path('/kaggle/input/datasets/ojassangwai/mlchallengedata/train/train_ground_truth.tsv'),
            Path('/kaggle/input/datasets/ojassangwai/mlchallengedata/train_ground_truth.tsv'),
        ]
        for gp in gt_candidates:
            if gp.exists():
                GT_FILE = gp
                break
    # Diagnostics if still not found
    if GT_FILE is None:
        print('\n❌ Ground truth NOT FOUND! Listing all .tsv files under /kaggle/input:')
        for p in sorted(Path('/kaggle/input').rglob('*.tsv')):
            print(f'    {p}')
        raise FileNotFoundError(
            'train_ground_truth.tsv not found anywhere under /kaggle/input.\n'
            'FIX: Attach the competition dataset (mlchallengedata) via + Add Input.'
        )
    DATA_DIR = GT_FILE.parent
else:
    WORK_DIR = Path('work')
    CAND_DIR = WORK_DIR
    CAND_FILE = CAND_DIR / 'candidates_train.parquet'
    NORM_DIR = WORK_DIR / 'normalized'
    NORM_FORMAT = 'parquet'
    DATA_DIR = Path('data/student_resource/dataset')
    GT_FILE = DATA_DIR / 'train' / 'train_ground_truth.tsv'

OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Candidates File: {CAND_FILE}")
print(f"Norm Dir:        {NORM_DIR} (format: {NORM_FORMAT})")
print(f"Data Dir:        {DATA_DIR}")
print(f"GT File:         {GT_FILE}")
print(f"Work Dir:        {WORK_DIR}")


In [ ]:
# ============================================================
# PRE-FLIGHT DEPENDENCY & ENVIRONMENT VALIDATION
# Runs in <1 second. Catches missing imports, broken paths,
# and API issues BEFORE the 5+ minute data loading phase.
# ============================================================

_checks_passed = 0
_checks_failed = 0
_errors = []

def _check(description, test_fn):
    global _checks_passed, _checks_failed
    try:
        result = test_fn()
        if result:
            print(f"  \u2705 {description}")
            _checks_passed += 1
        else:
            print(f"  \u274c {description}")
            _checks_failed += 1
            _errors.append(description)
    except Exception as e:
        print(f"  \u274c {description}: {e}")
        _checks_failed += 1
        _errors.append(f"{description}: {e}")

print("=" * 60)
print(" PRE-FLIGHT DEPENDENCY CHECKS")
print("=" * 60)

# --- 1. Core library imports ---
print("\n\U0001f4e6 Core Libraries:")
_check('numpy', lambda: np is not None)
_check('pandas', lambda: pd is not None)
_check('lightgbm', lambda: lgb is not None)

# --- 2. RapidFuzz components (the exact things used in feature engineering) ---
print("\n\U0001f4e6 RapidFuzz Components:")
_check('rapidfuzz.fuzz', lambda: callable(getattr(fuzz, 'ratio', None)))
_check('rapidfuzz.distance', lambda: distance is not None)
_check('rapidfuzz.distance.JaroWinkler', lambda: callable(getattr(JaroWinkler, 'similarity', None)))
_check('JaroWinkler.similarity smoke test', lambda: isinstance(JaroWinkler.similarity('hello', 'hallo'), float))

# --- 3. File path validation ---
print("\n\U0001f4c1 Required Files & Paths:")
_check(f'Candidates file: {CAND_FILE}', lambda: CAND_FILE is not None and CAND_FILE.exists())
_check(f'Norm directory: {NORM_DIR}', lambda: NORM_DIR is not None and NORM_DIR.exists())
_check(f'Ground truth file: {GT_FILE}', lambda: GT_FILE is not None and GT_FILE.exists())

if NORM_FORMAT == 'parquet':
    for _fn in ['train_s1_norm.parquet', 'train_s2_norm.parquet', 'train_s3_norm.parquet']:
        _fp = NORM_DIR / _fn
        _check(f'Normalized file: {_fn}', lambda _fp=_fp: _fp.exists())

# --- 4. Compute configuration ---
print("\n\u2699\ufe0f Compute Configuration:")
_check(f'Device: {device_param}', lambda: True)
if USE_GPU:
    _check(f'GPU: {n_gpus}x {gpu_name}', lambda: True)

# --- 5. Quick candidates parquet schema check ---
print("\n\U0001f50d Quick Schema Check (candidates parquet header):")
try:
    import pyarrow.parquet as pq
    _pf = pq.ParquetFile(str(CAND_FILE))
    _schema_cols = _pf.schema_arrow.names
    _required_cols = ['s1_id', 'candidate_id', 'sim_name', 'sim_addr', 'route_name', 'route_addr']
    for _rc in _required_cols:
        _check(f'Column "{_rc}" in candidates', lambda _rc=_rc: _rc in _schema_cols)
    _check(f'Total rows: {_pf.metadata.num_rows:,}', lambda: _pf.metadata.num_rows > 0)
    del _pf
except Exception as e:
    print(f"  \u26a0\ufe0f Could not validate parquet schema: {e}")

# --- Summary ---
print("\n" + "=" * 60)
if _checks_failed == 0:
    print(f"\u2705 ALL {_checks_passed} CHECKS PASSED — safe to proceed with data loading.")
else:
    print(f"\u274c {_checks_failed} CHECK(S) FAILED out of {_checks_passed + _checks_failed}:")
    for _err in _errors:
        print(f"   \u2022 {_err}")
    raise RuntimeError(
        f"{_checks_failed} dependency check(s) failed! Fix these BEFORE running data loading. "
        f"This saved you ~5 minutes of wasted GPU/CPU time."
    )
print("=" * 60)

# Cleanup validation variables
del _checks_passed, _checks_failed, _errors, _check


In [ ]:
# ============================================================
# LOAD DATA — flat-array design, no Python dict-of-dicts
# ============================================================
# Memory design:
#  - Normalized records stored as FLAT PARALLEL NUMPY ARRAYS
#    indexed by integer. One str->int dict for ID lookups.
#    ~350 bytes/record vs ~1500 bytes for Python dict-of-dicts.
#  - df_cands is NOT loaded here; read in 5M-row chunks later.
#  - Peak RAM during this cell: ~4-5 GB.

import re, unicodedata
import pyarrow.parquet as pq
import psutil

def _mem_gb():
    return psutil.Process().memory_info().rss / 1e9

t0 = time.time()
print(f'  RAM at start: {_mem_gb():.1f} GB')

def _quick_normalize(name, addr):
    name = str(name) if pd.notna(name) else ''
    addr = str(addr) if pd.notna(addr) else ''
    name = unicodedata.normalize('NFKD', name)
    name = re.sub(r'[^a-zA-Z0-9\s]', ' ', name.lower())
    name = re.sub(r'\s+', ' ', name).strip()
    addr = unicodedata.normalize('NFKD', addr)
    addr = re.sub(r'[^a-zA-Z0-9\s]', ' ', addr.lower())
    addr = re.sub(r'\s+', ' ', addr).strip()
    return name, addr

def _load_raw_tsv_as_norm(tsv_path):
    print(f'    [Fallback] Normalizing from raw TSV: {tsv_path}')
    df_raw = pd.read_csv(tsv_path, sep='\t', dtype=str)
    rows = []
    for _, r in df_raw.iterrows():
        nc, ac = _quick_normalize(r.get('business_name', ''), r.get('business_address', ''))
        rows.append({'entity_id': str(r['entity_id']).strip(),
                     'name_core': nc, 'addr_core': ac,
                     'house_number': '', 'state_code': ''})
    return pd.DataFrame(rows)

# === 1. Load normalized DataFrames ===
print('Loading normalized records...')
_need_cols = ['entity_id', 'name_core', 'addr_core', 'house_number', 'state_code']

if NORM_FORMAT == 'parquet':
    df_s1 = pd.read_parquet(NORM_DIR / 'train_s1_norm.parquet')
    df_s2 = pd.read_parquet(NORM_DIR / 'train_s2_norm.parquet')
    df_s3 = pd.read_parquet(NORM_DIR / 'train_s3_norm.parquet')
elif NORM_FORMAT == 'raw_tsv':
    print('  [Fallback] Building from raw TSVs...')
    df_s1 = _load_raw_tsv_as_norm(NORM_DIR / 'train' / 'train_source1.tsv')
    df_s2 = _load_raw_tsv_as_norm(NORM_DIR / 'train' / 'train_source2.tsv')
    df_s3 = _load_raw_tsv_as_norm(NORM_DIR / 'train' / 'train_source3.tsv')
else:
    raise FileNotFoundError('Cannot find normalized parquets OR raw TSVs.')

# Drop unneeded columns before concat to keep memory tight
_s1_have  = [c for c in _need_cols if c in df_s1.columns]
_s2_have  = [c for c in _need_cols if c in df_s2.columns]
_s3_have  = [c for c in _need_cols if c in df_s3.columns]
df_s1     = df_s1[_s1_have]
df_target = pd.concat([df_s2[_s2_have], df_s3[_s3_have]], ignore_index=True)
del df_s2, df_s3; gc.collect()
print(f'  S1: {len(df_s1):,} | Target (S2+S3): {len(df_target):,} | RAM: {_mem_gb():.1f} GB')

# === 2. Ground truth ===
print(f'Loading ground truth from: {GT_FILE}')
df_gt = pd.read_csv(GT_FILE, sep='\t', dtype=str)
gt_map = {}
for sid, raw_m in zip(
    df_gt['source1_entity_id'].astype(str).str.strip(),
    df_gt['matched_entity_ids'].fillna('').astype(str)
):
    if raw_m and raw_m != 'nan':
        gt_map[sid] = frozenset(m.strip() for m in raw_m.split(',') if m.strip())
    else:
        gt_map[sid] = frozenset()
del df_gt; gc.collect()
total_singletons = sum(1 for v in gt_map.values() if len(v) == 0)
print(f'  GT entities: {len(gt_map):,} | Singletons: {total_singletons:,} ({total_singletons/len(gt_map)*100:.1f}%)')

# === 3. Build FLAT LOOKUP ARRAYS (no dict-of-dicts) ===
# Parallel arrays indexed by integer position.
# One str->int mapping dict per side for ID resolution.
# Memory: ~350 bytes/record vs ~1500 bytes for nested Python dicts.
print('Building flat lookup arrays...')

def _col(df, name, default=''):
    return df[name].fillna(default).values if name in df.columns else np.full(len(df), default, dtype=object)

# S1 parallel arrays
s1_ids_arr   = df_s1['entity_id'].values
s1_name_arr  = _col(df_s1, 'name_core')
s1_addr_arr  = _col(df_s1, 'addr_core')
s1_house_arr = _col(df_s1, 'house_number')
s1_state_arr = _col(df_s1, 'state_code')
# Single int-indexed dict (str->int, NOT str->dict)
s1_id_to_idx = {eid: i for i, eid in enumerate(s1_ids_arr)}

# Target parallel arrays
tgt_ids_arr   = df_target['entity_id'].values
tgt_name_arr  = _col(df_target, 'name_core')
tgt_addr_arr  = _col(df_target, 'addr_core')
tgt_house_arr = _col(df_target, 'house_number')
tgt_state_arr = _col(df_target, 'state_code')
tgt_id_to_idx = {eid: i for i, eid in enumerate(tgt_ids_arr)}

del df_s1, df_target; gc.collect()
print(f'  S1 lookup: {len(s1_id_to_idx):,} | Tgt lookup: {len(tgt_id_to_idx):,} | RAM: {_mem_gb():.1f} GB')
print(f'  All lookup structures ready in {time.time()-t0:.1f}s')
print(f'  (df_cands will be loaded in 5M-row chunks during feature engineering)')


In [ ]:
# ============================================================
# FEATURE ENGINEERING — chunked pyarrow batch processing
# ============================================================
# Reads candidates.parquet in 5M-row batches via pyarrow.
# Never holds all 84.7M pairs in memory simultaneously.
# Peak RAM during this cell: ~6 GB total.

print('\n' + '=' * 50)
print(' FEATURE ENGINEERING (chunked)')
print('=' * 50)

CHUNK_SIZE = 5_000_000
_pf = pq.ParquetFile(str(CAND_FILE))
n_pairs_total = _pf.metadata.num_rows
print(f'  Total pairs: {n_pairs_total:,} | Chunk size: {CHUNK_SIZE:,}')
print(f'  RAM before chunked processing: {_mem_gb():.1f} GB')

def compute_token_jaccard(tok1, s2):
    """tok1 is a cached frozenset; s2 is a raw string (split inline)."""
    if not tok1 or not s2:
        return 0.0
    tok2 = set(s2.split())
    if not tok2:
        return 0.0
    inter = len(tok1 & tok2)
    return inter / (len(tok1) + len(tok2) - inter)

_MISS = -1          # sentinel for entity IDs not found in lookup
_EMPTY = frozenset()
chunk_results = []  # list of dicts of numpy arrays, one per chunk
n_processed = 0
t_feat = time.time()

for _batch in _pf.iter_batches(batch_size=CHUNK_SIZE):
    _df = _batch.to_pandas()
    n_c = len(_df)

    # Extract chunk arrays, then immediately free the DataFrame
    c_s1_ids    = _df['s1_id'].values
    c_cand_ids  = _df['candidate_id'].values
    c_name_sim  = _df['sim_name'].values.astype(np.float32)
    c_addr_sim  = _df['sim_addr'].values.astype(np.float32)
    c_route_name = _df['route_name'].values.astype(np.int8)
    c_route_addr = _df['route_addr'].values.astype(np.int8)
    del _df; gc.collect()

    # Resolve entity IDs to integer indices in one vectorized pass
    s1_idx  = np.fromiter((s1_id_to_idx.get(x, _MISS)  for x in c_s1_ids),  dtype=np.int32, count=n_c)
    tgt_idx = np.fromiter((tgt_id_to_idx.get(x, _MISS) for x in c_cand_ids), dtype=np.int32, count=n_c)

    # Allocate output arrays for this chunk
    jw_sim         = np.zeros(n_c, dtype=np.float32)
    name_jaccard   = np.zeros(n_c, dtype=np.float32)
    name_exact     = np.zeros(n_c, dtype=np.int8)
    name_len_diff  = np.zeros(n_c, dtype=np.float32)
    first_token_eq = np.zeros(n_c, dtype=np.int8)
    house_match    = np.zeros(n_c, dtype=np.int8)
    state_match    = np.zeros(n_c, dtype=np.int8)
    addr_jaccard   = np.zeros(n_c, dtype=np.float32)
    is_source2     = np.zeros(n_c, dtype=np.int8)
    labels         = np.zeros(n_c, dtype=np.int8)

    # Per-S1-entity cache — amortizes S1 lookups across ~38 candidates per entity
    _prev_s1i  = _MISS
    _s1_name   = ''
    _s1_first  = ''
    _tok1      = _EMPTY
    _addr_tok1 = _EMPTY
    _h1        = ''
    _st1       = ''
    _true_set  = _EMPTY

    for i in range(n_c):
        s1i = s1_idx[i]
        tgi = tgt_idx[i]

        # Refresh S1 cache on entity change
        if s1i != _prev_s1i:
            _prev_s1i = s1i
            if s1i >= 0:
                _s1_name   = s1_name_arr[s1i]
                _s1_first  = _s1_name.split()[0] if _s1_name else ''
                _tok1      = frozenset(_s1_name.split())
                _addr_tok1 = frozenset(s1_addr_arr[s1i].split())
                _h1        = s1_house_arr[s1i]
                _st1       = s1_state_arr[s1i]
                _true_set  = gt_map.get(c_s1_ids[i], _EMPTY)
            else:
                _s1_name = _s1_first = _h1 = _st1 = ''
                _tok1 = _addr_tok1 = _true_set = _EMPTY

        # Source indicator (cheap string prefix check)
        is_source2[i] = 1 if c_cand_ids[i][:2] == 'S2' else 0

        if tgi >= 0:
            t_name = tgt_name_arr[tgi]

            if _s1_name and t_name:
                if _s1_name == t_name:
                    jw_sim[i] = 1.0; name_exact[i] = 1; name_jaccard[i] = 1.0
                else:
                    jw_sim[i]       = JaroWinkler.similarity(_s1_name, t_name)
                    name_jaccard[i] = compute_token_jaccard(_tok1, t_name)
                name_len_diff[i]  = abs(len(_s1_name) - len(t_name))
                t_first = t_name.split()[0] if t_name else ''
                first_token_eq[i] = 1 if (_s1_first and t_first and _s1_first == t_first) else 0

            h2 = tgt_house_arr[tgi]
            if _h1 and h2:
                house_match[i] = 1 if _h1 == h2 else -1

            st2 = tgt_state_arr[tgi]
            if _st1 and st2:
                state_match[i] = 1 if _st1 == st2 else -1

            addr_jaccard[i] = compute_token_jaccard(_addr_tok1, tgt_addr_arr[tgi])

        labels[i] = 1 if c_cand_ids[i] in _true_set else 0

    chunk_results.append({
        's1_id':          c_s1_ids,
        'candidate_id':   c_cand_ids,
        'name_sim':       c_name_sim,
        'addr_sim':       c_addr_sim,
        'route_name':     c_route_name,
        'route_addr':     c_route_addr,
        'route_both':     (c_route_name & c_route_addr).astype(np.int8),
        'jw_sim':         jw_sim,
        'name_jaccard':   name_jaccard,
        'name_exact':     name_exact,
        'name_len_diff':  name_len_diff,
        'first_token_eq': first_token_eq,
        'house_match':    house_match,
        'state_match':    state_match,
        'addr_jaccard':   addr_jaccard,
        'is_source2':     is_source2,
        'label':          labels,
    })
    del jw_sim, name_jaccard, name_exact, name_len_diff, first_token_eq
    del house_match, state_match, addr_jaccard, is_source2, labels
    del s1_idx, tgt_idx

    n_processed += n_c
    elapsed = time.time() - t_feat
    qps = n_processed / elapsed
    eta = (n_pairs_total - n_processed) / qps if qps > 0 else 0
    print(f'  {n_processed:,} / {n_pairs_total:,} | {elapsed:.0f}s | {qps:,.0f} pairs/s | ETA {eta:.0f}s | RAM {_mem_gb():.1f} GB')

del _pf
print(f'\n  Feature loop done in {time.time()-t_feat:.1f}s | RAM: {_mem_gb():.1f} GB')

# Free the large lookup structures from Cell 3 immediately (saves ~5-7 GB RAM)
print('  Freeing record lookup arrays...')
del s1_ids_arr, s1_name_arr, s1_addr_arr, s1_house_arr, s1_state_arr, s1_id_to_idx
del tgt_ids_arr, tgt_name_arr, tgt_addr_arr, tgt_house_arr, tgt_state_arr, tgt_id_to_idx
gc.collect()
print(f'  Lookup structures freed | RAM: {_mem_gb():.1f} GB')

# Concatenate all chunks into final DataFrame
print('  Concatenating chunks...')
df_features = pd.DataFrame({
    col: np.concatenate([c[col] for c in chunk_results])
    for col in chunk_results[0].keys()
})
del chunk_results; gc.collect()
print(f'  df_features: {df_features.shape} | RAM: {_mem_gb():.1f} GB')

# Interaction & context features (fully vectorized)
print('  Computing group interaction features...')
df_features['sim_product'] = (df_features['name_sim'] * df_features['addr_sim']).astype(np.float32)
df_features['sim_max']     = np.maximum(df_features['name_sim'], df_features['addr_sim']).astype(np.float32)

s1_cat      = pd.Categorical(df_features['s1_id'])
grouped_max = df_features.groupby(s1_cat, observed=True)['sim_max'].transform('max')
df_features['gap_to_top']   = (grouped_max - df_features['sim_max']).astype(np.float32)
df_features['cand_rank']    = df_features.groupby(s1_cat, observed=True)['sim_max'].rank(
    ascending=False, method='min').astype(np.int16)
df_features['n_candidates'] = df_features.groupby(s1_cat, observed=True)['sim_max'].transform('count').astype(np.int16)
del s1_cat, grouped_max; gc.collect()

pos_count = int(df_features['label'].sum())
neg_count = len(df_features) - pos_count
print(f'\n  Total pairs:  {len(df_features):,}')
print(f'  Positives:    {pos_count:,} ({pos_count/len(df_features)*100:.2f}%)')
print(f'  Negatives:    {neg_count:,}')
print(f'  Imbalance:    1:{neg_count//max(pos_count,1)}')
print(f'  RAM final:    {_mem_gb():.1f} GB')


In [ ]:
# ============================================================
# GROUPED 2-FOLD CROSS-VALIDATION (FAST TRACK FOR SUBMISSION)
# 2 stratified folds: 100% OOF coverage with 4x less training compute.
# ============================================================

print("\n" + "=" * 50)
print(" GROUPED 2-FOLD CROSS-VALIDATION")
print("=" * 50)

N_FOLDS = 2
SEED = 42

# Get unique S1 entities from the feature DataFrame
unique_s1 = df_features['s1_id'].unique()
n_entities = len(unique_s1)
print(f"  Unique S1 entities in candidates: {n_entities:,}")

# Compute cardinality stratum per entity for stratification
strata = np.zeros(n_entities, dtype=np.int32)
for i, sid in enumerate(unique_s1):
    m_count = len(gt_map.get(sid, set()))
    if m_count == 0:
        strata[i] = 0    # Singleton
    elif m_count <= 2:
        strata[i] = 1
    elif m_count <= 5:
        strata[i] = 2
    else:
        strata[i] = 3

# Stratified fold assignment
fold_map = {}
rng = np.random.default_rng(SEED)
for s_val in np.unique(strata):
    indices = np.where(strata == s_val)[0]
    rng.shuffle(indices)
    for fold, fold_idx in enumerate(np.array_split(indices, N_FOLDS)):
        for idx in fold_idx:
            fold_map[unique_s1[idx]] = fold

# Assign fold to every row (cast to int8 to save RAM)
df_features['fold'] = df_features['s1_id'].map(fold_map).astype(np.int8)
del fold_map, strata, unique_s1; gc.collect()

for f in range(N_FOLDS):
    n_val = int((df_features['fold'] == f).sum())
    print(f"  Fold {f}: {n_val:,} pairs ({n_val/len(df_features)*100:.1f}%)")

# Checkpoint metadata (s1_id, candidate_id, label, fold) to disk
meta_path = WORK_DIR / 'cand_meta_train.parquet'
print(f'\n  Check pointing pair metadata to {meta_path}...')
df_features[['s1_id', 'candidate_id', 'label', 'fold']].to_parquet(meta_path, index=False)
print(f'  Saved metadata checkpoint ({meta_path.stat().st_size / 1e6:.1f} MB) | RAM: {_mem_gb():.1f} GB')

# Immediately drop string columns from df_features
df_features.drop(columns=['s1_id', 'candidate_id'], inplace=True)
gc.collect()
print(f'  Dropped string columns from df_features | RAM: {_mem_gb():.1f} GB')


In [ ]:
# ============================================================
# LIGHTGBM MODEL TRAINING (FAST & STABLE CONFIGURATION)
# Optimized for fast convergence and zero assertion crashes.
# ============================================================

print("\n" + "=" * 50)
print(" LIGHTGBM MODEL TRAINING (FAST TRACK)")
print("=" * 50)

FEATURE_COLUMNS = [
    'name_sim', 'addr_sim', 'route_name', 'route_addr', 'route_both',
    'jw_sim', 'name_jaccard', 'name_exact', 'name_len_diff', 'first_token_eq',
    'house_match', 'state_match', 'addr_jaccard', 'is_source2',
    'sim_product', 'sim_max', 'gap_to_top', 'cand_rank', 'n_candidates'
]

LGBM_PARAMS = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,        # 2x faster convergence (150 trees vs 475)
    'num_leaves': 31,
    'max_bin': 127,             # 2x faster histogram construction
    'min_child_samples': 200,   # Prevents serial_tree_learner right_count assertion crash
    'min_split_gain': 1e-4,     # Prunes micro-splits for speed & stability
    'feature_fraction': 1.0,    # All 19 features evaluated; prevents empty binary splits
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'n_estimators': 1000,       # Early stopping will stop around ~150-200
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1,
}

# Verify GPU device capability with a lightweight smoke test
if USE_GPU:
    try:
        _dtest = lgb.Dataset(np.zeros((10, 2), dtype=np.float32), label=np.zeros(10, dtype=np.int8))
        _tparams = {'objective': 'binary', 'device': 'cuda', 'verbose': -1, 'min_data_in_bin': 1}
        _b = lgb.train(_tparams, _dtest, num_boost_round=1)
        del _dtest, _b
        LGBM_PARAMS['device'] = 'cuda'
        print("  LightGBM GPU acceleration verified: ENABLED (device='cuda')")
    except Exception as e:
        try:
            _dtest = lgb.Dataset(np.zeros((10, 2), dtype=np.float32), label=np.zeros(10, dtype=np.int8))
            _tparams = {'objective': 'binary', 'device': 'gpu', 'verbose': -1, 'min_data_in_bin': 1}
            _b = lgb.train(_tparams, _dtest, num_boost_round=1)
            del _dtest, _b
            LGBM_PARAMS['device'] = 'gpu'
            print("  LightGBM OpenCL GPU acceleration verified: ENABLED (device='gpu')")
        except Exception as e2:
            print(f"  [Notice] LightGBM GPU build not supported ({e}). Using optimized multithreaded CPU.")
            USE_GPU = False
            LGBM_PARAMS.pop('device', None)
else:
    print("  Configured LightGBM to use multithreaded CPU")

# Extract labels and folds
n_rows = len(df_features)
y_all = df_features['label'].values.astype(np.int8)
folds = df_features['fold'].values.astype(np.int8)

# Construct contiguous float32 feature matrix, deleting columns as we transfer
print(f"  Allocating X_all (float32, {n_rows:,} x {len(FEATURE_COLUMNS)})...")
X_all = np.empty((n_rows, len(FEATURE_COLUMNS)), dtype=np.float32)
for j, col in enumerate(FEATURE_COLUMNS):
    X_all[:, j] = df_features[col].values
    del df_features[col]

del df_features; gc.collect()
print(f"  X_all ready ({X_all.nbytes / 1e9:.1f} GB) | RAM: {_mem_gb():.1f} GB")

oof_preds = np.zeros(n_rows, dtype=np.float32)
models = []

for fold in range(N_FOLDS):
    t_fold = time.time()
    
    val_mask = (folds == fold)
    train_mask = ~val_mask
    
    print(f"\n  --- Fold {fold+1}/{N_FOLDS} [Train: {int(np.sum(train_mask)):,} | Val: {int(np.sum(val_mask)):,}] ---")
    print(f"  RAM before Dataset: {_mem_gb():.1f} GB")
    
    X_tr = X_all[train_mask]
    y_tr = y_all[train_mask]
    X_va = X_all[val_mask]
    y_va = y_all[val_mask]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURE_COLUMNS, free_raw_data=True)
    dval = lgb.Dataset(X_va, label=y_va, reference=dtrain, feature_name=FEATURE_COLUMNS, free_raw_data=True)
    
    del X_tr, y_tr, X_va, y_va; gc.collect()
    print(f"  RAM after Dataset: {_mem_gb():.1f} GB")
    
    callbacks = [
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(period=50)
    ]
    
    booster = lgb.train(
        LGBM_PARAMS,
        dtrain,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        callbacks=callbacks
    )
    models.append(booster)
    
    # OOF predictions on validation set
    val_preds = booster.predict(X_all[val_mask], num_iteration=booster.best_iteration)
    oof_preds[val_mask] = val_preds
    
    y_va_fold = y_all[val_mask]
    val_logloss = -np.mean(y_va_fold * np.log(val_preds + 1e-15) + (1 - y_va_fold) * np.log(1 - val_preds + 1e-15))
    elapsed_fold = time.time() - t_fold
    print(f"  Fold {fold+1}/{N_FOLDS}: best_iter={booster.best_iteration}, "
          f"val_logloss={val_logloss:.5f}, time={elapsed_fold:.1f}s | RAM: {_mem_gb():.1f} GB")
    
    # Save model checkpoint
    model_path = WORK_DIR / f'lgbm_model_fold{fold}.txt'
    booster.save_model(str(model_path))
    
    del dtrain, dval, val_preds, y_va_fold; gc.collect()

# Release feature matrix and fold labels
del X_all, folds; gc.collect()
print(f"\n  All {N_FOLDS} folds complete | RAM: {_mem_gb():.1f} GB")


In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 50)
print(" FEATURE IMPORTANCE (avg across folds)")
print("=" * 50)

avg_importance = np.zeros(len(FEATURE_COLUMNS))
for m in models:
    avg_importance += m.feature_importance(importance_type='gain')
avg_importance /= len(models)

imp_df = pd.DataFrame({'feature': FEATURE_COLUMNS, 'importance': avg_importance})
imp_df = imp_df.sort_values('importance', ascending=False)
for _, row in imp_df.iterrows():
    bar = '█' * int(row['importance'] / imp_df['importance'].max() * 30)
    print(f"  {row['feature']:20s} {row['importance']:12.1f}  {bar}")

In [ ]:
# ============================================================
# SAVE OOF SCORES — score_C_train.parquet
# (Methodology A6: out-of-fold predictions for threshold tuning)
# ============================================================

print("\n" + "=" * 50)
print(" SAVING OOF SCORES")
print("=" * 50)

meta_path = WORK_DIR / 'cand_meta_train.parquet'
print(f'  Loading metadata checkpoint from {meta_path}...')
df_meta = pd.read_parquet(meta_path, columns=['s1_id', 'candidate_id', 'label'])

df_oof = pd.DataFrame({
    's1_id':        df_meta['s1_id'],
    'candidate_id': df_meta['candidate_id'],
    'score_C':      oof_preds,
    'label':        df_meta['label'],
})
del df_meta; gc.collect()

oof_path = WORK_DIR / 'score_C_train.parquet'
df_oof.to_parquet(oof_path, index=False)
print(f"  Saved: {oof_path} ({oof_path.stat().st_size / 1e6:.1f} MB, {len(df_oof):,} pairs)")

# Quick OOF quality check
pos_scores = oof_preds[y_all == 1]
neg_scores = oof_preds[y_all == 0]
print(f"\n  OOF Score Distribution:")
print(f"    Positives: mean={np.mean(pos_scores):.4f}, median={np.median(pos_scores):.4f}, p10={np.percentile(pos_scores, 10):.4f}")
print(f"    Negatives: mean={np.mean(neg_scores):.4f}, median={np.median(neg_scores):.4f}, p90={np.percentile(neg_scores, 90):.4f}")
print(f"    Separation: {np.mean(pos_scores) - np.mean(neg_scores):.4f}")
del pos_scores, neg_scores, y_all; gc.collect()


In [ ]:
# ============================================================
# QUICK MACRO F0.5 EVALUATION ON OOF
# (Methodology 2.5: threshold tuning preview)
# ============================================================

print("\n" + "=" * 50)
print(" OOF MACRO F0.5 EVALUATION")
print("=" * 50)

def compute_entity_f05(pred_ids, true_ids):
    if len(true_ids) == 0:
        return 1.0 if len(pred_ids) == 0 else 0.0
    if len(pred_ids) == 0:
        return 0.0
    tp = len(pred_ids & true_ids)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_ids)
    recall = tp / len(true_ids)
    denom = 0.25 * precision + recall
    return (1.25 * precision * recall) / denom if denom > 0 else 0.0

# Filter to candidates with score_C >= 0.25 to save massive RAM & execution time
# Candidates below 0.25 will never meet thresholds in [0.30, 0.90]
print('  Filtering candidate predictions (score_C >= 0.25)...')
plausible_mask = (df_oof['score_C'] >= 0.25)
df_sub = df_oof[plausible_mask][['s1_id', 'candidate_id', 'score_C']]
del df_oof, oof_preds; gc.collect()
print(f'  Plausible candidate pairs: {len(df_sub):,} | RAM: {_mem_gb():.1f} GB')

# Group OOF scores by entity
print("  Grouping candidate predictions by entity...")
grouped_oof = {}
s1_vals = df_sub['s1_id'].values
cand_vals = df_sub['candidate_id'].values
score_vals = df_sub['score_C'].values

for i in range(len(df_sub)):
    sid = str(s1_vals[i])
    if sid not in grouped_oof:
        grouped_oof[sid] = []
    grouped_oof[sid].append((str(cand_vals[i]), float(score_vals[i])))

del df_sub, s1_vals, cand_vals, score_vals; gc.collect()

# Sort each entity's candidates descending
for sid in grouped_oof:
    grouped_oof[sid].sort(key=lambda x: -x[1])

# Scan a few thresholds to find approximate best
print("  Scanning thresholds...")
best_f05 = -1.0
best_th = 0.5
for th in [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
    entity_scores = []
    for sid in gt_map:
        cands = grouped_oof.get(sid, [])
        pred = {c for c, s in cands if s >= th}
        entity_scores.append(compute_entity_f05(pred, gt_map[sid]))
    f05 = np.mean(entity_scores)
    pred_empty = sum(1 for sid in gt_map if not any(s >= th for _, s in grouped_oof.get(sid, [])))
    print(f"    t={th:.2f}: F0.5={f05:.4f}, singletons_pred={pred_empty:,}")
    if f05 > best_f05:
        best_f05 = f05
        best_th = th

print(f"\n  Best single threshold: t={best_th:.2f}, F0.5={best_f05:.4f}")
print("  (Two-threshold optimization will be done in NB 05)")
